In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key was found!')
elif not api_key.startswith('sk-proj-'):
    print('An API key was found, but it doesnt start with "sk-proj-", i.e, it is not an OpenAI API key')
elif api_key.strip() != api_key:
    print('An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them.')
else:
    print('The API key found and looks good so far!')


The API key found and looks good so far!


# Google ADK

The book's first example (pp. 226-229) is the pattern in its plainest form: two agents that differ only in their model, a cheap one for simple questions and an expensive one for hard ones, and a router in front that picks per query. The router's metric is deliberately crude: fewer than 20 words goes to the cheap model. A Critic Agent's system prompt follows, for reviewing answers. The book labels the code "Conceptual Python-like structure, not runnable code", and it is not:

| The book | Here |
|---|---|
| `gemini-2.5-pro` | `404 NOT_FOUND`, "no longer available to new users"; `gemini-pro-latest` |
| `gemini-2.5-flash` | `gemini-flash-latest`, as in the other notebooks |
| `context.current_message.text` | no such attribute; `context.user_content.parts[0].text` |
| `await gemini_flash_agent.run_async(context.current_message)` | `run_async` takes the invocation context and yields events; the router declares both agents as `sub_agents` and passes their events through |
| `yield Event(content=f"Flash Agent processed: {response}")` | `Event.content` must be a `types.Content`; a string raises a `ValidationError` (chapter 7's bug) |
| `AsyncGenerator` used, never imported | imported |
| `CRITIC_SYSTEM_PROMPT` defined, never used | a critic agent with that prompt reviews the router's last answer |
| no runner, no query | two queries, one short and one long, and a run helper that prints the tokens and seconds each call cost |

The cost lines are the point of this chapter: watch the **thinking** tokens. Gemini Pro thinks before it answers, and on the long query it spends several times more tokens thinking than answering.

The next cell holds the router, the critic prompt and the run helper, with no model. The Gemini cell and the Ollama cell each build the agents and run them, so either can run on its own.

In [3]:
# Book listings: Chapter 16, Hands-On Code Example, pp. 226-229 - two agents on different
# models, a QueryRouterAgent, and a Critic Agent's system prompt. The book marks the first
# two "Conceptual Python-like structure, not runnable code"; this cell is the runnable form.
# It holds the router, the critic prompt and a run helper, and no model. The Gemini cell
# and the Ollama cell below each build the agents and call run_router_demo.
import time
from typing import AsyncGenerator

from google.adk.agents import Agent, BaseAgent
from google.adk.agents.invocation_context import InvocationContext
from google.adk.events import Event
from google.adk.runners import InMemoryRunner
from google.genai import types


class QueryRouterAgent(BaseAgent):
    """Routes user queries to the appropriate LLM agent based on complexity.

    sub_agents is [cheap_agent, capable_agent]; the book referenced two globals instead.
    """

    async def _run_async_impl(self, context: InvocationContext) -> AsyncGenerator[Event, None]:
        user_query = context.user_content.parts[0].text  # book: context.current_message.text
        query_length = len(user_query.split())  # Simple metric: number of words
        cheap_agent, capable_agent = self.sub_agents

        if query_length < 20:  # Example threshold for simplicity vs. complexity
            print(f"Routing to {cheap_agent.name} for short query (length: {query_length})")
            target = cheap_agent
        else:
            print(f"Routing to {capable_agent.name} for long query (length: {query_length})")
            target = capable_agent
        # book: response = await agent.run_async(context.current_message), then
        # Event(content=f"... {response}"). run_async takes the context and yields events,
        # and Event.content must be a types.Content, so the sub-agent's events pass through.
        async for event in target.run_async(context):
            yield event


CRITIC_SYSTEM_PROMPT = """
You are the **Critic Agent**, serving as the quality assurance arm of our collaborative research assistant system. Your primary function is to **meticulously review and challenge** information from the Researcher Agent, guaranteeing **accuracy, completeness, and unbiased presentation**.

Your duties encompass:
* **Assessing research findings** for factual correctness, thoroughness, and potential leanings.
* **Identifying any missing data** or inconsistencies in reasoning.
* **Raising critical questions** that could refine or expand the current understanding.
* **Offering constructive suggestions** for enhancement or exploring different angles.
* **Validating that the final output is comprehensive** and balanced.

All criticism must be constructive. Your goal is to fortify the research, not invalidate it. Structure your feedback clearly, drawing attention to specific points for revision. Your overarching aim is to ensure the final research product meets the highest possible quality standards.
"""

QUERIES = [
    "What is the capital of Australia?",
    "Compare routing every request to one large model with routing each request by its "
    "complexity, considering cost, latency, answer quality and the risk of sending a hard "
    "question to a weak model. Answer in at most three sentences.",
]


async def ask(agent, query: str) -> str:
    """Runs one query in a fresh session; prints the answer and what it cost."""
    runner = InMemoryRunner(agent=agent, app_name="routing_app")
    session = await runner.session_service.create_session(app_name="routing_app", user_id="user_1")
    message = types.Content(role="user", parts=[types.Part(text=query)])
    started, answer = time.perf_counter(), ""
    async for event in runner.run_async(user_id="user_1", session_id=session.id, new_message=message):
        if event.content and event.content.parts:
            answer += "".join(part.text or "" for part in event.content.parts)
        if event.usage_metadata:
            usage = event.usage_metadata
            print(f"  [usage] {event.author}: {usage.prompt_token_count} prompt + "
                  f"{usage.candidates_token_count} answer + {usage.thoughts_token_count or 0} thinking tokens")
    print(f"  [time] {time.perf_counter() - started:.1f} s")
    print(f"--- {answer.strip()}")
    return answer


async def run_router_demo(router, critic) -> None:
    """Sends each query through the router, then has the critic review the last answer."""
    for query in QUERIES:
        print(f"\n>>> {query}")
        answer = await ask(router, query)
    print("\n>>> Critic review of the last answer")
    await ask(critic, f"Question:\n{QUERIES[-1]}\n\nAnswer to review:\n{answer}")

In [4]:
# The book's two agents on Gemini Pro and Flash, behind the router, plus a critic.
from google.genai.types import GenerateContentConfig, AutomaticFunctionCallingConfig

# ADK runs its own tool loop and leaves the genai SDK's automatic function calling enabled,
# which logs a one-time warning. Disabling it removes the warning.
NO_AFC = GenerateContentConfig(automatic_function_calling=AutomaticFunctionCallingConfig(disable=True))

# Agent using the more expensive Gemini Pro 2.5
gemini_pro_agent = Agent(
    name="GeminiProAgent",
    model="gemini-pro-latest",  # book: gemini-2.5-pro, no longer available to new users
    description="A highly capable agent for complex queries.",
    instruction="You are an expert assistant for complex problem-solving.",
    generate_content_config=NO_AFC,
)

# Agent using the less expensive Gemini Flash 2.5
gemini_flash_agent = Agent(
    name="GeminiFlashAgent",
    model="gemini-flash-latest",  # book: gemini-2.5-flash
    description="A fast and efficient agent for simple queries.",
    instruction="You are a quick assistant for straightforward questions.",
    generate_content_config=NO_AFC,
)

router = QueryRouterAgent(
    name="QueryRouter",
    description="Routes user queries to the appropriate LLM agent based on complexity.",
    sub_agents=[gemini_flash_agent, gemini_pro_agent],
)
critic = Agent(name="CriticAgent", model="gemini-flash-latest", instruction=CRITIC_SYSTEM_PROMPT,
               generate_content_config=NO_AFC)

await run_router_demo(router, critic)


>>> What is the capital of Australia?
Routing to GeminiFlashAgent for short query (length: 6)


  [usage] GeminiFlashAgent: 49 prompt + 7 answer + 61 thinking tokens
  [time] 3.6 s
--- The capital of Australia is Canberra.

>>> Compare routing every request to one large model with routing each request by its complexity, considering cost, latency, answer quality and the risk of sending a hard question to a weak model. Answer in at most three sentences.
Routing to GeminiProAgent for long query (length: 38)


  [usage] GeminiProAgent: 86 prompt + 90 answer + 663 thinking tokens
  [time] 7.5 s
--- Routing every request to one large model guarantees consistently high answer quality and eliminates the risk of sending complex questions to weak models, but it results in maximum cost and higher latency. Conversely, routing requests by complexity significantly lowers average costs and latency by directing easier tasks to smaller, faster models. However, this efficient approach introduces a critical risk: if the router misclassifies a hard query as simple, the weaker model will likely produce a low-quality answer.

>>> Critic review of the last answer


  [usage] CriticAgent: 341 prompt + 423 answer + 496 thinking tokens
  [time] 9.8 s
--- ### **Review and Quality Assessment**

**Strengths:**
* **Constraint Compliance:** The draft strictly follows the length requirement, consisting of exactly three clear and coherent sentences.
* **Coverage of Core Dimensions:** It successfully addresses all four mandated evaluation criteria:
  1. **Cost:** Contrasts the "maximum cost" of the monolithic approach with "significantly lowers average costs."
  2. **Latency:** Highlights "higher latency" against faster response times via smaller models.
  3. **Quality:** Balances "consistently high answer quality" with the risk of "low-quality answers."
  4. **Routing Risk:** Directly captures the trade-off between eliminating routing risk entirely versus the risk of classifier error/misrouting.

---

### **Critical Observations & Minor Refinements**

1. **Routing Overhead (Latency Nuance):** 
   While complexity routing lowers *average generation latency*

## Google ADK + Ollama

In [5]:
# Same router and critic, on two local models served by Ollama through LiteLLM: a 1.5B model
# as the cheap tier, a 7B model as the capable one.
# Prerequisites:
#   ollama pull qwen2.5:1.5b
#   ollama pull qwen2.5:7b
#   litellm, from the google-adk[extensions] extra (uv sync, then restart the kernel)
# Run cell 4 first (the shared cell under "# Google ADK"); this cell needs nothing else: not the Gemini cell, and no Google key.
import os

from google.adk.models.lite_llm import LiteLlm

os.environ.setdefault("OLLAMA_API_BASE", "http://localhost:11434")

ollama_large_agent = Agent(
    name="QwenLargeAgent",
    model=LiteLlm(model="ollama_chat/qwen2.5:7b"),
    description="A highly capable agent for complex queries.",
    instruction="You are an expert assistant for complex problem-solving.",
)
ollama_small_agent = Agent(
    name="QwenSmallAgent",
    model=LiteLlm(model="ollama_chat/qwen2.5:1.5b"),
    description="A fast and efficient agent for simple queries.",
    instruction="You are a quick assistant for straightforward questions.",
)

ollama_router = QueryRouterAgent(
    name="QueryRouter",
    description="Routes user queries to the appropriate LLM agent based on complexity.",
    sub_agents=[ollama_small_agent, ollama_large_agent],
)
ollama_critic = Agent(name="CriticAgent", model=LiteLlm(model="ollama_chat/qwen2.5:7b"),
                      instruction=CRITIC_SYSTEM_PROMPT)

await run_router_demo(ollama_router, ollama_critic)


>>> What is the capital of Australia?
Routing to QwenSmallAgent for short query (length: 6)


  [usage] QwenSmallAgent: 59 prompt + 8 answer + 0 thinking tokens
  [time] 0.9 s
--- The capital of Australia is Canberra.

>>> Compare routing every request to one large model with routing each request by its complexity, considering cost, latency, answer quality and the risk of sending a hard question to a weak model. Answer in at most three sentences.
Routing to QwenLargeAgent for long query (length: 38)


  [usage] QwenLargeAgent: 95 prompt + 125 answer + 0 thinking tokens
  [time] 2.5 s
--- Routing every request to one large model simplifies the system architecture and can reduce latency since there's no need to determine the model's suitability for each query, but it may lead to suboptimal answers for simpler queries and could be less cost-effective as the large model might be overpowered for many tasks. Routing by complexity can enhance cost efficiency and answer quality by matching queries to models that are well-suited to their complexity, thereby improving response accuracy and potentially reducing the computational resources needed. However, this approach increases the risk of harder questions being sent to less capable models, which could negatively impact answer quality and system reliability.

>>> Critic review of the last answer


  [usage] CriticAgent: 382 prompt + 189 answer + 0 thinking tokens
  [time] 4.3 s
--- The answer provided effectively compares the two routing strategies, highlighting key advantages and disadvantages. However, to enhance clarity and depth, consider the following refinements:

- **Simplified System Architecture**: Routing every request to one large model simplifies the system architecture, reducing latency as there's no need to determine the model's suitability for each query. This, however, can result in suboptimal answers for simpler queries and might be less cost-effective due to the overpowered nature of the large model for many tasks.

- **Cost-Efficiency and Answer Quality**: Routing by complexity enhances cost efficiency and answer quality by matching queries to models that are well-suited to their complexity, thus improving response accuracy and potentially reducing computational resources. However, this approach increases the risk of harder questions being sent to less capable

# OpenAI

The book's second example (pp. 230-233) routes by meaning instead of length. A classifier model labels each prompt `simple`, `reasoning` or `internet_search`, and each label has its own model: a small one for simple questions, a reasoning model for multi-step ones, and for current events a Google search whose results are pasted into the prompt (chapter 14's RAG, as one branch of a router).

| The book | Here |
|---|---|
| model names hard-coded in each function | the same names in `CLASSIFIER_MODEL` and `MODELS`, set by the OpenAI and Ollama cells; `client` likewise |
| `o4-mini` for `reasoning` | `404`, "has been deprecated"; `gpt-5-mini`, OpenAI's current small reasoning model |
| Google Custom Search with `GOOGLE_CUSTOM_SEARCH_API_KEY` + `GOOGLE_CSE_ID` | kept, and used when both keys are set. Google closed that API to new customers and shuts it down on 1 January 2027, so without the keys a one-result `mock_search` stands in |
| `raise ValueError` unless all three keys are set | only the OpenAI cell checks its key, so the Ollama cell needs none |
| one test prompt, two commented out | all three, one per route |
| emoji in the `print` calls | removed |

The classifier's reply goes straight into `json.loads`. That works because the models answer with bare JSON as told; a reply wrapped in a code fence would raise `JSONDecodeError`.

In [6]:
# Book listing: Chapter 16, Hands-On Code with OpenAI, pp. 230-233.
# This cell holds the classifier, the search, the response step and the router, and no
# model. The functions read three globals when they run: `client`, `CLASSIFIER_MODEL` and
# `MODELS` (the book hard-codes OpenAI and its model names). The OpenAI cell and the Ollama
# cell below each bind those three, then call run_router_prompts.
import json
import os

import requests

GOOGLE_CUSTOM_SEARCH_API_KEY = os.getenv("GOOGLE_CUSTOM_SEARCH_API_KEY")
GOOGLE_CSE_ID = os.getenv("GOOGLE_CSE_ID")


# --- Step 1: Classify the Prompt ---
def classify_prompt(prompt: str) -> dict:
    system_message = {
        "role": "system",
        "content": (
            "You are a classifier that analyzes user prompts and returns one of three categories ONLY:\n\n"
            "- simple\n"
            "- reasoning\n"
            "- internet_search\n\n"
            "Rules:\n"
            "- Use 'simple' for direct factual questions that need no reasoning or current events.\n"
            "- Use 'reasoning' for logic, math, or multi-step inference questions.\n"
            "- Use 'internet_search' if the prompt refers to current events, recent data, or things not in your training data.\n\n"
            "Respond ONLY with JSON like:\n"
            '{ "classification": "simple" }'
        ),
    }

    user_message = {"role": "user", "content": prompt}

    response = client.chat.completions.create(
        model=CLASSIFIER_MODEL, messages=[system_message, user_message], temperature=1  # book: "gpt-4o"
    )

    reply = response.choices[0].message.content
    return json.loads(reply)


# --- Step 2: Google Search ---
def google_search(query: str, num_results=1) -> list:
    url = "https://www.googleapis.com/customsearch/v1"
    params = {
        "key": GOOGLE_CUSTOM_SEARCH_API_KEY,
        "cx": GOOGLE_CSE_ID,
        "q": query,
        "num": num_results,
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        results = response.json()

        if "items" in results and results["items"]:
            return [
                {
                    "title": item.get("title"),
                    "snippet": item.get("snippet"),
                    "link": item.get("link"),
                }
                for item in results["items"]
            ]
        else:
            return []
    except requests.exceptions.RequestException as e:
        return {"error": str(e)}


# Google closed the Custom Search JSON API to new customers and shuts it down on
# 1 January 2027, so most students cannot get the two keys above. Without them the router
# uses this one-result mock instead, so the internet_search path still runs.
def mock_search(query: str, num_results=1) -> list:
    print(f"  [mock_search] {query!r}")
    return [{
        "title": "Australian Open 2026 key dates",
        "snippet": "The Australian Open 2026 main draw starts on Sunday 18 January 2026 at "
                   "Melbourne Park. The women's final is on 31 January, the men's on 1 February.",
        "link": "https://ausopen.com",
    }]


search = google_search if GOOGLE_CUSTOM_SEARCH_API_KEY and GOOGLE_CSE_ID else mock_search


# --- Step 3: Generate Response ---
def generate_response(prompt: str, classification: str, search_results=None) -> str:
    model = MODELS[classification]  # book: one hard-coded model name per branch
    if classification == "simple":
        full_prompt = prompt
    elif classification == "reasoning":
        full_prompt = prompt
    elif classification == "internet_search":
        # Convert each search result dict to a readable string
        if search_results:
            search_context = "\n".join(
                [
                    f"Title: {item.get('title')}\nSnippet: {item.get('snippet')}\nLink: {item.get('link')}"
                    for item in search_results
                ]
            )
        else:
            search_context = "No search results found."
        full_prompt = f"""Use the following web results to answer the user query:

{search_context}

Query: {prompt}"""

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": full_prompt}],
        temperature=1,
    )

    return response.choices[0].message.content, model


# --- Step 4: Combined Router ---
def handle_prompt(prompt: str) -> dict:
    classification_result = classify_prompt(prompt)
    classification = classification_result["classification"]

    search_results = None
    if classification == "internet_search":
        search_results = search(prompt)  # book: google_search(prompt)

    answer, model = generate_response(prompt, classification, search_results)
    return {"classification": classification, "response": answer, "model": model}


# The book runs the first prompt and leaves the other two commented out; one per route.
TEST_PROMPTS = [
    "What is the capital of Australia?",
    "Explain the impact of quantum computing on cryptography.",
    "When does the Australian Open 2026 start, give me full date?",
]


def run_router_prompts() -> None:
    """Routes each test prompt and prints the book's three lines (emoji removed)."""
    for test_prompt in TEST_PROMPTS:
        print(f"\n>>> {test_prompt}")
        result = handle_prompt(test_prompt)
        print("Classification:", result["classification"])
        print("Model Used:", result["model"])
        print("Response:\n", result["response"])

In [7]:
# The book's router on OpenAI: gpt-4o classifies, then gpt-4o-mini, a reasoning model or
# gpt-4o answers. The book's reasoning model, o4-mini, is retired (404 "has been
# deprecated"); gpt-5-mini is its current small reasoning counterpart.
from openai import OpenAI

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is not set; run the Ollama cell below instead.")

client = OpenAI()
CLASSIFIER_MODEL = "gpt-4o"
MODELS = {"simple": "gpt-4o-mini", "reasoning": "gpt-5-mini", "internet_search": "gpt-4o"}

run_router_prompts()


>>> What is the capital of Australia?


Classification: simple
Model Used: gpt-4o-mini
Response:
 The capital of Australia is Canberra.

>>> Explain the impact of quantum computing on cryptography.


Classification: reasoning
Model Used: gpt-5-mini
Response:
 Short answer
- Large, fault-tolerant quantum computers would break the public-key algorithms that secure most Internet traffic today (RSA, DH, ECC) by running Shor’s algorithm.
- Symmetric ciphers and hash functions are not broken, but Grover’s algorithm gives a quadratic speedup against brute force / preimage search, so recommended symmetric key lengths should be doubled (e.g., use 256-bit keys rather than 128-bit for long-term secrecy).
- The practical response is migration to quantum-resistant (post-quantum) cryptography, improving crypto agility, and planning for “record now, decrypt later” threats.

What the quantum algorithms do (high level)
- Shor’s algorithm: factors large integers and solves discrete log in polynomial time on a sufficiently large, error-corrected quantum computer. That directly undermines RSA, classic Diffie–Hellman, and elliptic-curve schemes (ECDH, ECDSA).
- Grover’s algorithm: gives a sqrt(N) speed

  [mock_search] 'When does the Australian Open 2026 start, give me full date?'


Classification: internet_search
Model Used: gpt-4o
Response:
 The Australian Open 2026 starts on Sunday, 18 January 2026.


## OpenAI + Ollama

In [8]:
# Same router on local models. Ollama serves an OpenAI-compatible API at /v1, so the book's
# OpenAI client works unchanged: only base_url and the model names differ.
# Prerequisites:
#   ollama pull qwen2.5:1.5b
#   ollama pull qwen2.5:7b
# Run cell 9 first (the shared cell under "# OpenAI"); this cell needs nothing else: not the OpenAI cell, and no OpenAI key.
# There is no local reasoning model here, so the reasoning route uses the 7B model.
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")  # the key is ignored
CLASSIFIER_MODEL = "qwen2.5:7b"
MODELS = {"simple": "qwen2.5:1.5b", "reasoning": "qwen2.5:7b", "internet_search": "qwen2.5:7b"}

run_router_prompts()


>>> What is the capital of Australia?


Classification: simple
Model Used: qwen2.5:1.5b
Response:
 The capital of Australia is Canberra.

>>> Explain the impact of quantum computing on cryptography.


Classification: reasoning
Model Used: qwen2.5:7b
Response:
 Quantum computing has the potential to significantly impact cryptography by introducing new challenges and opportunities, particularly in the realm of public-key cryptography. Here’s a detailed explanation of how quantum computing affects cryptographic systems:

1. **Shor's Algorithm**: The most significant impact of quantum computing comes from Shor's algorithm, which was developed by Peter Shor in 1994. This quantum algorithm can efficiently factor large integers, which is critical for the security of many cryptographic systems today, especially those based on the RSA public-key cryptosystem. Factoring large integers is an NP-complete problem, which is extremely difficult for classical computers, but Shor's algorithm can solve it in polynomial time on a quantum computer. This means that once a sufficiently large quantum computer (a quantum computer with many qubits) is available, current RSA and some discrete logarithm-based

  [mock_search] 'When does the Australian Open 2026 start, give me full date?'


Classification: internet_search
Model Used: qwen2.5:7b
Response:
 The Australian Open 2026 main draw starts on Sunday, January 18, 2026.


# OpenRouter

The book's third example (pp. 233-235) moves model selection out of your code: OpenRouter is one OpenAI-compatible endpoint in front of hundreds of models. `"model": "openrouter/auto"` lets it choose per prompt, and `"models": [...]` gives it a fallback list to try in order when a model is down, rate-limited or filtered. The reply's `model` field says which one answered.

The listing needs an OpenRouter key with paid credit, so the cell skips without `OPENROUTER_API_KEY` and its committed output is only that message. The book's fallback list starts with `anthropic/claude-3.5-sonnet`, which OpenRouter no longer lists; `anthropic/claude-haiku-4.5` takes its place.

The Ollama cell sends the same request to Ollama, which speaks the same API: only the URL, key and model change. That is the practical lesson here. An OpenAI-compatible endpoint makes the model a configuration value.

In [9]:
# Book listing: Chapter 16, Hands-On Code Example (OpenRouter), pp. 233-235.
# This cell holds the book's request as a function, and no model. The OpenRouter cell and
# the Ollama cell below each send it to their own OpenAI-compatible endpoint.
import json
import os

import requests


def chat_completion(url: str, api_key: str, body: dict) -> None:
    """The book's requests.post, with the reply printed: which model answered, and what."""
    response = requests.post(
        url=url,
        headers={
            "Authorization": f"Bearer {api_key}",
            "HTTP-Referer": "<YOUR_SITE_URL>",  # Optional. Site URL for rankings on openrouter.ai.
            "X-Title": "<YOUR_SITE_NAME>",  # Optional. Site title for rankings on openrouter.ai.
        },
        data=json.dumps(body),
    )
    reply = response.json()
    if "error" in reply:
        print(f"  [{body.get('model', body.get('models'))}] error: {reply['error']}")
        return None
    print(f"  [model] {reply['model']}")
    print(f"--- {reply['choices'][0]['message']['content'].strip()}")
    return reply


MESSAGES = [
    {
        "role": "user",
        "content": "What is the meaning of life?"
    }
]

In [10]:
# The book's three OpenRouter requests: a fixed model, automatic model selection, and a
# sequential fallback list. Needs OPENROUTER_API_KEY (https://openrouter.ai/keys, paid
# credit); without it the cell says so and skips. Not run for the committed output.
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"

if not OPENROUTER_API_KEY:
    print("Skipped: set OPENROUTER_API_KEY to send these requests to OpenRouter.")
else:
    print(">>> A fixed model")
    chat_completion(OPENROUTER_URL, OPENROUTER_API_KEY, {"model": "openai/gpt-4o", "messages": MESSAGES})
    print("\n>>> Automated model selection: OpenRouter picks, the reply names its choice")
    chat_completion(OPENROUTER_URL, OPENROUTER_API_KEY, {"model": "openrouter/auto", "messages": MESSAGES})
    print("\n>>> Sequential model fallback: the first model that answers wins")
    # book: anthropic/claude-3.5-sonnet, no longer listed on OpenRouter
    chat_completion(OPENROUTER_URL, OPENROUTER_API_KEY,
                    {"models": ["anthropic/claude-haiku-4.5", "gryphe/mythomax-l2-13b"], "messages": MESSAGES})

Skipped: set OPENROUTER_API_KEY to send these requests to OpenRouter.


## OpenRouter + Ollama

In [11]:
# The same request sent to Ollama's OpenAI-compatible endpoint: only the URL, the key and
# the model name change. Ollama has neither "openrouter/auto" nor a server-side fallback
# list, so the fallback is a client-side loop: the first model is not pulled, so it fails
# and the next one answers.
# Prerequisites:
#   ollama pull qwen2.5:1.5b
#   ollama pull qwen2.5:7b
# Run cell 14 first (the shared cell under "# OpenRouter"); this cell needs nothing else, and no API key.
OLLAMA_URL = "http://localhost:11434/v1/chat/completions"

print(">>> A fixed model")
chat_completion(OLLAMA_URL, "ollama", {"model": "qwen2.5:7b", "messages": MESSAGES})

print("\n>>> Sequential model fallback, done by the client")
for model in ["llama3.3:70b", "qwen2.5:1.5b"]:
    if chat_completion(OLLAMA_URL, "ollama", {"model": model, "messages": MESSAGES}):
        break

>>> A fixed model


  [model] qwen2.5:7b
--- The meaning of life is a philosophical question that has puzzled thinkers and scholars for centuries. Unlike scientific or factual questions, the meaning of life is deeply personal and can vary widely from one individual to another. Here are a few perspectives:

1. **Philosophical Perspectives**: Philosophers might argue that the meaning of life is to pursue happiness, knowledge, or existential freedom. For example, Aristotle believed in achieving eudaimonia, often translated as 'flourishing or living well', which involves virtuous and meaningful activities.

2. **Religious Perspectives**: In many religions, the meaning of life is often explained in terms of divine purpose, achieving salvation, or serving a higher power, such as in Christianity where it is to love God and one's neighbors, or in Islam where it is to worship Allah and live a good life.

3. **Personal Perspectives**: For many people, the meaning of life is found in relationships, personal growth, 

  [model] qwen2.5:1.5b
--- As an AI assistant, I must inform you that there is no universally accepted definition of the meaning of life, as the concept of "meaning" can be subjective and varies greatly from person to person. The question of what is the "meaning" of life has been a subject of philosophical, religious, and scientific inquiry for centuries. Here are a few potential, albeit speculative, examples of what some people believe the meaning of life to be:

1. Purpose: Some people believe that life has a particular purpose, such as achieving happiness, making a significant contribution to society, or experiencing meaningful relationships. 

2. Spiritual Connection: For many, the meaning of life is deeply connected to a spiritual or religious journey. They believe that there is a higher power or force that guides their lives, and that they have a particular purpose.

3. Personal Fulfillment: Others believe that life means finding happiness and fulfillment in their personal lives.